## 首元素归一化 ：

- 代码：u = v / v[0]

- 特点：第一个数是 1。

- 用途：直观查看相对比例。

In [21]:
import numpy as np
from numpy.linalg import eig, inv

def solve_mdof_first_element_norm(M, C, K, F_vector=None):
    """
    求解多自由度系统，使用 [首元素归一化] (First Element = 1)
    """
    n = len(M)
    
    # 1. 求解特征值问题 (M^-1 * K) * u = omega^2 * u
    eigenvalues, eigenvectors = eig(inv(M) @ K)
    
    # 排序：从低频到高频
    idx = np.argsort(eigenvalues)
    wn = np.sqrt(eigenvalues[idx])
    raw_modes = eigenvectors[:, idx]
    
    # ==============================================
    # 核心修改：首元素归一化 (Normalize to First Element = 1)
    # ==============================================
    normalized_modes = np.zeros_like(raw_modes)
    for i in range(n):
        vi = raw_modes[:, i]
        # 将整个向量除以第一个元素 vi[0]
        # 注意：在极少数特殊结构中，如果第一个元素本来就是0（节点），这种方法会失效
        if abs(vi[0]) < 1e-10:
            print(f"警告：第 {i+1} 阶模态的第一个元素接近 0，无法使用此归一化！改用最大值归一化。")
            normalized_modes[:, i] = vi / np.max(np.abs(vi))
        else:
            normalized_modes[:, i] = vi / vi[0] 
            
    # 2. 如果有外力，计算峰峰值 (可选)
    # 为了演示，这里假设计算频率等于最低固有频率
    if F_vector is not None:
        target_w = wn[0] 
        Z = -target_w**2 * M + 1j * target_w * C + K
        X = inv(Z) @ F_vector
        peak_to_peak = 2 * np.abs(X)
    
    # ==============================================
    # 打印结果
    # ==============================================
    print("="*60)
    print(f"{n}自由度系统求解结果 (首元素归一化: u[0] = 1)")
    print("="*60)
    for i in range(n):
        print(f"模态 {i+1} (频率: {wn[i]/(2*np.pi):.4f} Hz)(频率: {wn[i]:.4f} rad):")
        print(f"  - 振型: {normalized_modes[:, i]}")
        print(f"  - 归一化振型: {normalized_modes[:, i]}")
        # 验证
        print(f"  - 验证首项: {normalized_modes[0, i]:.1f}")
        print("-" * 30)
        
    if F_vector is not None:
        print(f"受迫振动峰峰值 (在 w1={wn[0]:.2f} rad/s 处):")
        print(f"  {peak_to_peak}")


In [22]:
# ==========================================
# 考试修改区：在这里输入你的推导结果
# ==========================================

# 以一个典型的三自由度（3-DOF）系统为例：三个质量块 m, 之间用 k 和 c 连接
m1, m2, m3 = 0.5, 0.7, 0.3
k1, k2, k3 = 200, 300, 170 
c1, c2, c3 = 0.1, 0.2, 0.08  # 阻尼项

# 1. 质量矩阵 M
M = np.array([
    [m1, 0,  0],
    [0,  m2, 0],
    [0,  0,  m3]
])

# 2. 刚度矩阵 K (根据你的 EOM 推导填写)
K = np.array([
    [k1+k2, -k2,    0],
    [-k2,    k2+k3, -k3],
    [0,     -k3,    k3]
])

# 3. 阻尼矩阵 C (如果没有阻尼，设为全 0 矩阵)
C = np.array([
    [c1+c2, -c2,    0],
    [-c2,    c2+c3, -c3],
    [0,     -c3,    c3]
])

# 运行求解
F_input = np.array([0, 0, 0.5]) 

solve_mdof_first_element_norm(M, C, K, F_input)

3自由度系统求解结果 (首元素归一化: u[0] = 1)
模态 1 (频率: 1.5534 Hz)(频率: 9.7606 rad):
  - 振型: [1.         1.50788413 1.81262802]
  - 归一化振型: [1.         1.50788413 1.81262802]
  - 验证首项: 1.0
------------------------------
模态 2 (频率: 4.2506 Hz)(频率: 26.7072 rad):
  - 振型: [ 1.          0.47787324 -1.84704974]
  - 归一化振型: [ 1.          0.47787324 -1.84704974]
  - 验证首项: 1.0
------------------------------
模态 3 (频率: 6.0176 Hz)(频率: 37.8094 rad):
  - 振型: [ 1.         -0.7159161   0.47015157]
  - 归一化振型: [ 1.         -0.7159161   0.47015157]
  - 验证首项: 1.0
------------------------------
受迫振动峰峰值 (在 w1=9.76 rad/s 处):
  [1.16783634 1.76096149 2.11688201]
